In [11]:
import os
import warnings
warnings.filterwarnings('ignore')

import torch
import whisper
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from pydub import AudioSegment
from pydub.silence import split_on_silence

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device.upper()}")
print()

Device: CPU



In [ ]:
# Load Whisper for English transcription
print("Loading Whisper model...")
whisper_model = whisper.load_model("small")  # Using 'small' for better accuracy
print("✓ Whisper loaded\n")

# Load translation model (NLLB for Telugu/Hindi)
print("Loading translation model...")
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
translation_model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
print("✓ Translation model loaded\n")

# Language codes for Telugu and Hindi
LANG_CODES = {
    'telugu': 'tel_Telu',
    'hindi': 'hin_Deva'
}

def audio_to_segments(audio_path):
    audio = AudioSegment.from_file(audio_path)
    
    # Split audio on silence for better transcription
    segments = split_on_silence(
        audio,
        min_silence_len=500,  # minimum silence length in ms
        silence_thresh=audio.dBFS - 14,  # silence threshold
        keep_silence=250  # keep some silence at edges
    )
    
    return segments

def transcribe_audio_segment(audio_segment):
    """Transcribe a single audio segment (from project1.ipynb)"""
    import time
    
    # Normalize audio volume
    audio_segment = audio_segment.normalize()
    
    # Export segment to temporary file
    temp_file = f"temp_segment_{time.time()}.wav"
    audio_segment.export(
        temp_file, 
        format="wav",
        parameters=["-ar", "16000", "-ac", "1"]  # 16kHz sample rate, mono
    )
    
    try:
        # Transcribe in English
        result = whisper_model.transcribe(
            temp_file,
            language='en',
            task='transcribe',
            fp16=False
        )
        english_text = result['text'].strip()
        
        if english_text:
            return english_text
        else:
            return None
                    
    except Exception as e:
        print(f"    Transcription error: {str(e)[:100]}")
        return None
                
    finally:
        # Clean up temp file
        try:
            time.sleep(0.1)
            if os.path.exists(temp_file):
                os.remove(temp_file)
        except:
            pass

def translate_text(text, output_language):
    """Translate English text to Telugu or Hindi using NLLB"""
    if not text or not text.strip():
        return ""
    
    target_code = LANG_CODES.get(output_language.lower())
    if not target_code:
        raise ValueError(f"Language must be 'telugu' or 'hindi', got: {output_language}")
    
    tokenizer.src_lang = "eng_Latn"
    
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    
    with torch.no_grad():
        generated = translation_model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(target_code),
            max_length=512,
            num_beams=5,
            early_stopping=True
        )
    
    translated_text = tokenizer.batch_decode(generated, skip_special_tokens=True)[0]
    return translated_text

def transcribe_and_translate(audio_file, output_language):
    print(f"Processing audio file: {audio_file}\n")
    
    # Step 1: Split audio into segments (better transcription quality)
    segments = audio_to_segments(audio_file)
    print(f"Found {len(segments)} segments\n")
    
    results = []
    
    # Step 2: Transcribe each segment
    for i, segment in enumerate(segments):
        print(f"Processing segment {i+1}/{len(segments)}...")
        english_text = transcribe_audio_segment(segment)
        
        if not english_text:
            print(f"  No speech detected in segment {i+1}\n")
            continue
        
        print(f"  English: {english_text}")
        results.append({
            'segment': i+1,
            'english_text': english_text
        })
        print()
    
    # Step 3: Combine all segments into complete transcript
    complete_transcript_english = " ".join([result['english_text'] for result in results])
    
    # Step 4: Translate complete transcript (better context preservation)
    print(f"\nTranslating to {output_language.capitalize()}...")
    complete_transcript_translated = translate_text(complete_transcript_english, output_language)
    print(f"✓ Translation complete\n")
    
    return {
        'english': complete_transcript_english,
        'translated': complete_transcript_translated,
        'segments': results,
        'language': output_language
    }

print("Ready to process audio!")

Loading Whisper model...
✓ Whisper loaded

Loading translation model...
✓ Translation model loaded

Ready to process audio!


In [13]:
# ===== CONFIGURATION =====
AUDIO_FILE = "audio_files/Julias the story.wav"
OUTPUT_LANGUAGE = "telugu"  # Change to 'telugu' or 'hindi'
# =========================

# Process the audio
result = transcribe_and_translate(AUDIO_FILE, OUTPUT_LANGUAGE)

# Display results
print("=" * 80)
print("RESULTS")
print("=" * 80)
print(f"\nEnglish Transcript:\n{result['english']}\n")
print(f"\n{OUTPUT_LANGUAGE.capitalize()} Transcript:\n{result['translated']}\n")

Processing audio file: audio_files/Julias the story.wav

Found 92 segments

Processing segment 1/92...
  English: Yeah, hello? Hello? Yes.

Processing segment 2/92...
  English: Look at three.

Processing segment 3/92...
  English: 2, 1, start. Elea stood in the center of shimmering glass atrium.

Processing segment 4/92...
  English: His chest swelling with pride so intense it felt like helium.

Processing segment 5/92...
  English: After 10 years of rejection, sleepless nights and a gear back cramp see

Processing segment 6/92...
  English: The Skyline project was finished.

Processing segment 7/92...
  English: It was his masterpiece.

Processing segment 8/92...
  English: He pulled out his phone.

Processing segment 9/92...
  English: his hands trembling with excitement and dialed the one person who had believed in him when no one else did.

Processing segment 10/92...
  English: is

Processing segment 11/92...
  English: and Strangle brother Julian. Julian. Elias Beamed.

Processi